# Celda experimental mínima — Evaluación de LLMs para soporte al cliente

**Trabajo Final — Juan Bautista Xifro — UAI 2026**

---

## Qué hace este notebook

Ejecuta el recorrido completo del experimento sobre una escala mínima:
**20 preguntas, 1 modelo, sin RAG**.

El objetivo no es obtener resultados publicables sino **validar que toda la cadena
funciona de punta a punta** antes de escalar. Cuando esto corre sin errores,
extender a 3 modelos × 2 condiciones es mecánico.

| Paso | Qué resuelve |
|---|---|
| 1. Setup | Instalación y credenciales |
| 2. Dataset | Carga y muestreo del corpus |
| 3. Modelo | Cliente unificado para cualquier proveedor |
| 4. Generación | Envío de consultas + medición de latencia y tokens |
| 5. Calidad | Cálculo de BERTScore y ROUGE |
| 6. Costo | Modelo de costo explícito y defendible |
| 7. Índice | Índice compuesto de la Hipótesis 3 |
| 8. Guardado | Persistencia de resultados |

> **Decisiones metodológicas fijadas aquí:** temperatura = 0 (reproducibilidad),
> semilla fija en el muestreo, y modelo de costo por amortización de cómputo
> para modelos open-source. Están documentadas en cada sección correspondiente.

---
## 1. Setup

### 1.1 Instalación

`bert-score` descarga un modelo de ~500 MB la primera vez. En Colab tarda 2-3 minutos.

In [ ]:
!pip install -q openai groq datasets bert-score rouge-score pandas tqdm

print('Instalación completa.')

### 1.2 Credenciales

**Groq** da acceso gratuito a LLaMA y Mistral (con límite de requests por minuto).
Registrate en `console.groq.com` y generá una API key.

**OpenAI** cobra por uso, pero GPT-4o mini es muy barato: evaluar 20 preguntas
cuesta menos de USD 0,01.

En Colab, guardá las keys en el panel de **Secrets** (ícono de llave 🔑) con los
nombres `GROQ_API_KEY` y `OPENAI_API_KEY`. Nunca las escribas directo en el código:
si compartís el notebook, quedan expuestas.

In [ ]:
import os

# --- En Google Colab: leer desde Secrets ---
try:
    from google.colab import userdata
    os.environ['GROQ_API_KEY']   = userdata.get('GROQ_API_KEY')
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print('Credenciales cargadas desde Colab Secrets.')
except Exception as e:
    # --- Fuera de Colab: definirlas como variables de entorno antes de correr ---
    print('No se detectó Colab. Asegurate de tener las variables de entorno definidas.')
    print('  export GROQ_API_KEY=...')
    print('  export OPENAI_API_KEY=...')

for k in ['GROQ_API_KEY', 'OPENAI_API_KEY']:
    estado = 'OK' if os.environ.get(k) else 'FALTA'
    print(f'  {k}: {estado}')

---
## 2. Dataset

Se usa **Bitext Customer Support LLM Chatbot Training Dataset**, disponible en
HuggingFace con licencia abierta. Contiene ~27.000 pares pregunta-respuesta de
soporte al cliente, etiquetados por categoría de intención.

Cada registro aporta:
- `instruction` → la consulta del cliente (entrada del modelo)
- `response` → la respuesta de referencia (contra la que se mide la calidad)
- `category` / `intent` → permiten analizar el desempeño por tipo de consulta

> ⚠️ **Limitación a documentar en la tesis:** este corpus está en inglés. Si el
> alcance del trabajo se define sobre PyMEs hispanohablantes, hay que traducir un
> subconjunto, buscar un corpus en español, o declarar explícitamente que el
> framework es agnóstico al idioma y que la validación se hizo en inglés.

In [ ]:
from datasets import load_dataset
import pandas as pd

SEMILLA = 42          # semilla fija -> muestra reproducible
N_MUESTRA = 20        # tamaño de la prueba mínima

ds = load_dataset('bitext/Bitext-customer-support-llm-chatbot-training-dataset',
                  split='train')
df_full = ds.to_pandas()

print(f'Corpus completo: {len(df_full):,} registros')
print(f'Columnas: {list(df_full.columns)}')
print(f'Categorías: {df_full["category"].nunique()}')

### 2.1 Muestreo estratificado

En lugar de tomar 20 registros al azar, se toma una muestra **estratificada por
categoría**. Así la prueba cubre distintos tipos de consulta y no queda sesgada
hacia una sola intención, que es lo que ocurriría con un muestreo simple sobre
un corpus desbalanceado.

In [ ]:
# Muestreo estratificado: al menos 1 registro por categoría, hasta completar N
muestra = (df_full
           .groupby('category', group_keys=False)
           .apply(lambda g: g.sample(1, random_state=SEMILLA))
           .reset_index(drop=True))

# Si faltan registros para llegar a N, completar al azar sin repetir
if len(muestra) < N_MUESTRA:
    restantes = df_full[~df_full.index.isin(muestra.index)]
    extra = restantes.sample(N_MUESTRA - len(muestra), random_state=SEMILLA)
    muestra = pd.concat([muestra, extra], ignore_index=True)

muestra = muestra.head(N_MUESTRA).reset_index(drop=True)

print(f'Muestra: {len(muestra)} registros, {muestra["category"].nunique()} categorías')
muestra[['category', 'instruction', 'response']].head(3)

---
## 3. Cliente unificado de modelos

Este es el punto de diseño más importante del notebook. En lugar de escribir
código distinto para cada proveedor, se define **una sola función** que recibe el
nombre del modelo y devuelve siempre la misma estructura de respuesta.

Gracias a eso, pasar de 1 modelo a 3 no requiere reescribir nada: solo agregar
entradas al diccionario `MODELOS`.

### Decisión metodológica: temperatura = 0

La temperatura controla la aleatoriedad de la generación. Con el valor por defecto,
cada corrida produce respuestas distintas y las pruebas estadísticas pierden validez,
porque la variación observada podría deberse al azar y no al modelo.
Se fija en **0** para que los resultados sean reproducibles.

In [ ]:
from openai import OpenAI
from groq import Groq
import time

TEMPERATURA = 0.0     # decisión metodológica: reproducibilidad
MAX_TOKENS  = 300     # respuestas de soporte: acotadas

# Catálogo de modelos. Para escalar el experimento, agregar entradas acá.
MODELOS = {
    'gpt-4o-mini': {
        'proveedor': 'openai',
        'id':        'gpt-4o-mini',
        'tipo':      'propietario',
    },
    'llama-3.1-8b': {
        'proveedor': 'groq',
        'id':        'llama-3.1-8b-instant',
        'tipo':      'open-source',
    },
    'mistral-saba': {
        'proveedor': 'groq',
        'id':        'mistral-saba-24b',
        'tipo':      'open-source',
    },
}

_clientes = {}

def _cliente(proveedor):
    """Devuelve (y cachea) el cliente del proveedor indicado."""
    if proveedor not in _clientes:
        if proveedor == 'openai':
            _clientes[proveedor] = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
        elif proveedor == 'groq':
            _clientes[proveedor] = Groq(api_key=os.environ['GROQ_API_KEY'])
        else:
            raise ValueError(f'Proveedor desconocido: {proveedor}')
    return _clientes[proveedor]

### 3.1 Prompt del sistema

El prompt se mantiene **idéntico para todos los modelos**. Si variara entre modelos,
no se podría atribuir la diferencia de calidad al modelo en sí, que es justamente
lo que las hipótesis se proponen medir.

In [ ]:
PROMPT_SISTEMA = (
    'Sos un agente de soporte al cliente. Respondé la consulta de forma clara, '
    'concisa y profesional. No inventes datos que no tengas. '
    'Respondé en el mismo idioma en que se te consulta.'
)

def generar(modelo_key, consulta, contexto=None):
    """Envía una consulta al modelo y devuelve respuesta + métricas operativas.

    Parámetros
    ----------
    modelo_key : clave del diccionario MODELOS
    consulta   : texto de la consulta del cliente
    contexto   : fragmentos recuperados por RAG. None = condición sin RAG.
                 (El parámetro ya existe para que agregar RAG después no
                  obligue a cambiar la firma de la función.)

    Devuelve
    --------
    dict con respuesta, latencia_s, tokens_in, tokens_out y error
    """
    cfg = MODELOS[modelo_key]
    cli = _cliente(cfg['proveedor'])

    user_msg = consulta if contexto is None else (
        f'Contexto relevante de la organización:\n{contexto}\n\n'
        f'Consulta del cliente: {consulta}'
    )

    mensajes = [
        {'role': 'system', 'content': PROMPT_SISTEMA},
        {'role': 'user',   'content': user_msg},
    ]

    t0 = time.perf_counter()
    try:
        r = cli.chat.completions.create(
            model=cfg['id'],
            messages=mensajes,
            temperature=TEMPERATURA,
            max_tokens=MAX_TOKENS,
        )
        latencia = time.perf_counter() - t0
        return {
            'respuesta':  r.choices[0].message.content.strip(),
            'latencia_s': round(latencia, 3),
            'tokens_in':  r.usage.prompt_tokens,
            'tokens_out': r.usage.completion_tokens,
            'error':      None,
        }
    except Exception as e:
        return {
            'respuesta':  None,
            'latencia_s': round(time.perf_counter() - t0, 3),
            'tokens_in':  0,
            'tokens_out': 0,
            'error':      f'{type(e).__name__}: {e}',
        }

### 3.2 Prueba de humo

Antes de correr las 20 consultas, se verifica que el modelo responde. Si esto falla,
el problema es de credenciales o de nombre de modelo, no del experimento.

In [ ]:
MODELO_PRUEBA = 'llama-3.1-8b'   # empezar con el gratuito

test = generar(MODELO_PRUEBA, '¿Cómo hago para cancelar mi pedido?')

if test['error']:
    print('FALLO:', test['error'])
else:
    print(f"Latencia: {test['latencia_s']} s | "
          f"tokens in/out: {test['tokens_in']}/{test['tokens_out']}\n")
    print(test['respuesta'])

---
## 4. Generación sobre la muestra

Se recorre la muestra enviando cada consulta al modelo. Se agrega una pausa entre
llamadas porque el tier gratuito de Groq limita los requests por minuto; sin la
pausa, las últimas consultas fallan por rate limit.

In [ ]:
from tqdm.auto import tqdm

PAUSA_S = 2.0   # respeta el rate limit del tier gratuito

def correr_experimento(modelo_key, df, contextos=None, pausa=PAUSA_S):
    """Ejecuta una celda experimental completa (un modelo, una condición)."""
    filas = []
    condicion = 'con_rag' if contextos is not None else 'sin_rag'

    for i, fila in tqdm(df.iterrows(), total=len(df),
                        desc=f'{modelo_key} [{condicion}]'):
        ctx = contextos[i] if contextos is not None else None
        out = generar(modelo_key, fila['instruction'], contexto=ctx)

        filas.append({
            'idx':        i,
            'modelo':     modelo_key,
            'tipo':       MODELOS[modelo_key]['tipo'],
            'condicion':  condicion,
            'category':   fila['category'],
            'consulta':   fila['instruction'],
            'referencia': fila['response'],
            **out,
        })
        time.sleep(pausa)

    return pd.DataFrame(filas)


resultados = correr_experimento(MODELO_PRUEBA, muestra)

n_err = resultados['error'].notna().sum()
print(f'\nCompletadas: {len(resultados) - n_err}/{len(resultados)}')
if n_err:
    print('Errores encontrados:')
    print(resultados[resultados['error'].notna()]['error'].value_counts())

---
## 5. Métricas de calidad

### BERTScore

Compara la respuesta generada con la de referencia usando **representaciones
semánticas**, no coincidencia de palabras. Esto importa en soporte al cliente,
donde "tu pedido fue cancelado" y "procedimos con la cancelación de tu orden"
significan lo mismo aunque compartan pocas palabras — una métrica léxica las
penalizaría injustamente.

Se reporta **F1**, que balancea precisión y exhaustividad.

### ROUGE-L

Se calcula como métrica complementaria, basada en la subsecuencia común más larga.
Sirve de contraste: si BERTScore y ROUGE-L divergen mucho, indica que el modelo
reformula con vocabulario distinto pero mantiene el significado.

In [ ]:
from bert_score import score as bertscore
from rouge_score import rouge_scorer

# Solo se evalúan las filas sin error
ok = resultados[resultados['error'].isna()].copy()

# --- BERTScore ---
P, R, F1 = bertscore(
    cands=ok['respuesta'].tolist(),
    refs=ok['referencia'].tolist(),
    lang='en',              # cambiar a 'es' si el corpus se traduce
    verbose=False,
)
ok['bertscore_f1'] = F1.numpy().round(4)

# --- ROUGE-L ---
rs = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
ok['rouge_l'] = [
    round(rs.score(ref, cand)['rougeL'].fmeasure, 4)
    for ref, cand in zip(ok['referencia'], ok['respuesta'])
]

print(f"BERTScore F1  media: {ok['bertscore_f1'].mean():.4f} "
      f"(desvío {ok['bertscore_f1'].std():.4f})")
print(f"ROUGE-L       media: {ok['rouge_l'].mean():.4f} "
      f"(desvío {ok['rouge_l'].std():.4f})")
print(f"Latencia      media: {ok['latencia_s'].mean():.2f} s")

---
## 6. Modelo de costo

**Este es el punto metodológicamente más delicado del trabajo.**

Para modelos propietarios el costo es directo: precio publicado por millón de tokens.

Para modelos open-source **el costo no es cero**. Aunque no se pague licencia, hay
consumo de cómputo. Comparar "costo de API" contra "cero" invalidaría la Hipótesis 2:
cualquier tribunal observaría que la comparación no es equivalente.

### Criterio adoptado

El costo de los modelos open-source se calcula como **amortización de una instancia
cloud con GPU** capaz de servirlos:

```
costo_por_consulta = (precio_hora_instancia / consultas_por_hora)
consultas_por_hora = 3600 / latencia_media_observada
```

Así ambas categorías se expresan en la misma unidad (USD por consulta) y la
comparación se sostiene. El precio de la instancia debe justificarse con una
cotización real y citarse en la tesis.

> Los valores de abajo son **placeholders**. Antes de la corrida definitiva hay que
> reemplazarlos por los precios vigentes al momento del experimento y documentar
> la fecha de consulta.

In [ ]:
# ─── Precios de referencia (USD) — VERIFICAR Y ACTUALIZAR ───
# Propietarios: precio por 1M de tokens, según tarifario del proveedor.
PRECIOS_API = {
    'gpt-4o-mini': {'in': 0.150, 'out': 0.600},   # USD por 1M tokens
}

# Open-source: precio por hora de la instancia cloud que los sirve.
PRECIO_INSTANCIA_HORA = 0.75   # USD/h — cotización a reemplazar


def costo_consulta(fila):
    """Costo en USD de una consulta individual."""
    m = fila['modelo']
    if m in PRECIOS_API:
        p = PRECIOS_API[m]
        return (fila['tokens_in']  / 1_000_000 * p['in'] +
                fila['tokens_out'] / 1_000_000 * p['out'])
    # Open-source: amortización por tiempo de cómputo efectivamente usado
    return PRECIO_INSTANCIA_HORA * (fila['latencia_s'] / 3600)


ok['costo_usd'] = ok.apply(costo_consulta, axis=1)
ok['costo_1000'] = ok['costo_usd'] * 1000

print(f"Costo por consulta : USD {ok['costo_usd'].mean():.6f}")
print(f"Costo por 1.000    : USD {ok['costo_1000'].mean():.4f}")

---
## 7. Índice compuesto (Hipótesis 3)

Es el aporte original del trabajo: un único indicador que integra las tres
dimensiones que condicionan la decisión organizacional.

**Construcción:**

1. Cada métrica se normaliza al rango [0, 1] con min-max scaling.
2. Costo y latencia se **invierten**, porque en ellas *menor es mejor*. Tras la
   inversión, en las tres dimensiones un valor más alto indica mejor desempeño.
3. Se promedian con ponderación equitativa (1/3 cada una), tal como establece
   la formulación de la hipótesis.

> ⚠️ La normalización min-max es **relativa al conjunto de modelos comparados**.
> Con un solo modelo el índice no tiene sentido — todos los valores dan 0 o 1.
> Recién se vuelve interpretable cuando se corren las 6 configuraciones.

In [ ]:
import numpy as np

def normalizar(s, invertir=False):
    """Min-max scaling a [0,1]. invertir=True para métricas donde menor es mejor."""
    rango = s.max() - s.min()
    if rango == 0:
        return pd.Series(0.5, index=s.index)   # sin variación: valor neutro
    n = (s - s.min()) / rango
    return 1 - n if invertir else n


def indice_compuesto(df, pesos=(1/3, 1/3, 1/3)):
    """Índice costo-calidad-latencia. Devuelve el df con las columnas agregadas."""
    w_cal, w_cos, w_lat = pesos
    d = df.copy()
    d['n_calidad']  = normalizar(d['bertscore_f1'])
    d['n_costo']    = normalizar(d['costo_usd'],  invertir=True)
    d['n_latencia'] = normalizar(d['latencia_s'], invertir=True)
    d['indice'] = (w_cal * d['n_calidad'] +
                   w_cos * d['n_costo'] +
                   w_lat * d['n_latencia']).round(4)
    return d


ok = indice_compuesto(ok)

resumen = ok.groupby(['modelo', 'condicion']).agg(
    n              = ('idx', 'count'),
    bertscore_f1   = ('bertscore_f1', 'mean'),
    rouge_l        = ('rouge_l', 'mean'),
    latencia_s     = ('latencia_s', 'mean'),
    costo_1000_usd = ('costo_1000', 'mean'),
    indice         = ('indice', 'mean'),
).round(4)

resumen

---
## 8. Guardado de resultados

Se persisten los resultados **a nivel de consulta individual**, no solo los promedios.
Esto es indispensable: las pruebas estadísticas que contrastan las hipótesis operan
sobre observaciones pareadas, y si solo se guardan promedios esa información se pierde
de forma irrecuperable.

In [ ]:
from datetime import datetime

sello = datetime.now().strftime('%Y%m%d_%H%M')
archivo = f'resultados_{MODELO_PRUEBA}_sin_rag_{sello}.csv'

ok.to_csv(archivo, index=False)
print(f'Guardado: {archivo}  ({len(ok)} filas)')

# En Colab, descargar el archivo:
# from google.colab import files; files.download(archivo)

---
## Próximos pasos

Si este notebook corrió sin errores, la cadena completa está validada.
El orden sugerido para avanzar:

**1. Completar la dimensión de modelos (sin RAG).**
Correr `correr_experimento()` para los tres modelos y concatenar los resultados.
Con eso ya se puede hacer una primera lectura de la Hipótesis 1 en su condición base.

```python
todos = pd.concat([correr_experimento(m, muestra) for m in MODELOS], ignore_index=True)
```

**2. Construir el módulo RAG.**
Indexar en FAISS un corpus de conocimiento derivado del dataset, recuperar los
fragmentos más relevantes por consulta y pasarlos al parámetro `contexto` que la
función `generar()` ya acepta. No hay que modificar nada del código existente.

**3. Escalar la muestra.**
Subir de 20 a 300-500 consultas. Es el mínimo razonable para que las pruebas
estadísticas tengan potencia suficiente. Recién ahí conviene incorporar Spark
para el preprocesamiento del corpus.

**4. Contrastar las hipótesis.**
Con los datos completos, verificar normalidad (Shapiro-Wilk) y, según el resultado,
aplicar la prueba paramétrica o no paramétrica que corresponda. Esta es la decisión
que quedó deliberadamente abierta en el Plan de TF.

**5. Dashboard.**
Streamlit sobre los CSV generados. Es lo último: sin datos no hay nada que mostrar.